# Expanding JSON-like Columns in Rider Information Data

In this notebook, we will clean and restructure the `rider_infos.csv` dataset.
The file contains two columns, **`pps`** and **`rdr`**, that store data in a JSON-like format (dictionary strings) inside each cell.
Our goal is to parse these columns, extract the nested key–value pairs, and expand them into separate columns.

This will make the dataset easier to analyze and work with — for example, turning entries like:

{‘One day races’: ‘2620’, ‘GC’: ‘5138’, ‘Time trial’: ‘2075’}

into individual columns:

| pps_One day races | pps_GC | pps_Time trial |
|--------------------|--------|----------------|
| 2620               | 5138   | 2075            |

We will:
1. Load the CSV into a pandas DataFrame.
2. Parse the `pps` and `rdr` columns into Python dictionaries.
3. Expand each dictionary into multiple columns with clear prefixes.
4. Merge everything back into one flat, clean DataFrame.
5. Save the final expanded dataset as a new CSV file.

In [ ]:
import pandas as pd
import json
import ast

# Load the CSV
file_path = "../data/data/rider_infos.csv"
df = pd.read_csv(file_path)

print("Initial DataFrame shape:", df.shape)
df.head()

In [ ]:
# Helper function to safely parse dictionary-like strings
def parse_dict(cell):
    if pd.isna(cell) or not isinstance(cell, str) or not cell.strip():
        return {}
    try:
        return ast.literal_eval(cell)
    except Exception:
        return {}

# Apply parsing to both columns
df['pps_parsed'] = df['pps'].apply(parse_dict)
df['rdr_parsed'] = df['rdr'].apply(parse_dict)

# Expand each parsed column into multiple new columns
pps_expanded = df['pps_parsed'].apply(pd.Series)
pps_expanded.columns = [f"pps_{col}" for col in pps_expanded.columns]

rdr_expanded = df['rdr_parsed'].apply(pd.Series)
rdr_expanded.columns = [f"rdr_{col}" for col in rdr_expanded.columns]

In [ ]:
# Merge everything together and drop old columns
df_flat = pd.concat([df.drop(columns=['pps', 'rdr', 'pps_parsed', 'rdr_parsed']),
                     pps_expanded, rdr_expanded], axis=1)

print("New DataFrame shape:", df_flat.shape)
df_flat.head()

In [ ]:
# Save the cleaned version
output_path = "../Joel/processed_data/rider_infos_expanded.csv"
df_flat.to_csv(output_path, index=False)

print(f"Expanded CSV saved to: {output_path}")

We now compare the original data to the processed data to see if there was any improvement in the missing data count.

In [ ]:
import matplotlib.pyplot as plt

def missing_data_summary(df, title="Missing Data Summary"):

    missing = df.isnull().mean() * 100
    missing = missing[missing > 0].sort_values(ascending=False)

    if not missing.empty:
        plt.figure(figsize=(10, 6))
        missing.plot.barh(color='skyblue', edgecolor='black')
        plt.title(title)
        plt.xlabel("Missing Data (%)")
        plt.ylabel("Columns")
        plt.gca().invert_yaxis()  # so the highest missing % appears at the top
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.show()
    else:
        print(f"No missing data in {title}.")

In [ ]:
missing_data_summary(df, "Missing Data Summary")